# Update Ground Truth JSON Files

This notebook updates the original ground truth JSON files by adding new accessions discovered during rule-based extraction.

**Workflow:**
1. Read `gt_discrepancies_detailed.csv` to get new accessions per PMCID
2. Classify each accession type (BioSample, SRA, WGS, RefSeq, etc.)
3. Format as `ACCESSION[TYPE]`
4. Add to `merge_accession_number` field in JSON
5. Save updated JSON

---

In [11]:
import os
import re
import json
import csv
from pathlib import Path
from typing import List, Dict, Set, Optional

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Configuration

In [12]:
# =============================================================================
# CONFIGURATION - Adjust paths as needed
# =============================================================================

DISCREPANCIES_CSV = "/content/drive/MyDrive/AI6129/gt_discrepancies_detailed.csv"
GROUND_TRUTH_FOLDER = "/content/drive/MyDrive/AI6129/ground_truth/"
OUTPUT_LOG = "/content/drive/MyDrive/AI6129/gt_update_log.csv"

# PMCIDs to exclude
EXCLUDED_PMCIDS = {"PMC9387262"}  # Outlier, already checked

## Accession Type Classification

In [13]:
def classify_accession_type(accession: str) -> str:
    """
    Classify an accession number and return its type suffix.
    """
    acc = accession.upper().strip()

    # BioSample
    if re.match(r'^SAM[NEDA]', acc):
        return "BioSample"

    # BioProject
    if re.match(r'^PRJ[NEDB]A?[0-9]+$', acc):
        return "BioProject"

    # SRA Run
    if re.match(r'^[SED]RR[0-9]+$', acc):
        return "SRA"

    # SRA Experiment
    if re.match(r'^[SED]RX[0-9]+$', acc):
        return "SRA"

    # SRA Sample
    if re.match(r'^[SED]RS[0-9]+$', acc):
        return "SRA"

    # SRA Study
    if re.match(r'^[SED]RP[0-9]+$', acc):
        return "SRA"

    # DDBJ
    if re.match(r'^DR[AXSP][0-9]+$', acc):
        return "DDBJ"

    # RefSeq
    if re.match(r'^(NC|NG|NM|NR|NP|NT|NW|NZ|XM|XR|XP|AP|YP|WP)_[0-9]+', acc):
        return "RefSeq"

    # GenBank Assembly
    if re.match(r'^GCA_[0-9]{9}', acc):
        return "Assembly"

    # RefSeq Assembly
    if re.match(r'^GCF_[0-9]{9}', acc):
        return "Assembly"

    # WGS (4-6 letters + 8-11 digits)
    if re.match(r'^[A-Z]{4,6}[0-9]{8,11}$', acc):
        return "WGS"

    # Protein (3 letters + 5 digits)
    if re.match(r'^[A-Z]{3}[0-9]{5}(\.[0-9]+)?$', acc):
        return "Protein"

    # GenBank nucleotide (1L5D)
    if re.match(r'^[A-Z]{1}[0-9]{5}$', acc):
        return "GenBank"

    # GenBank nucleotide (2L6D)
    if re.match(r'^[A-Z]{2}[0-9]{6}$', acc):
        return "GenBank"

    return "GenBank"


def format_accession_with_type(accession: str) -> str:
    """Format accession with type suffix: ACCESSION[TYPE]"""
    acc_type = classify_accession_type(accession)
    return f"{accession}[{acc_type}]"

In [14]:
# Test classification
test_accessions = [
    "SAMN08031253",    # BioSample
    "PRJNA448609",     # BioProject
    "SRR12967252",     # SRA
    "NC_001371",       # RefSeq
    "GCA_000001405.1", # Assembly
    "JAHCQI000000000", # WGS
    "AAR18814.1",      # Protein
    "AB123456",        # GenBank
]

print("Accession Type Classification Test:")
print("-" * 50)
for acc in test_accessions:
    print(f"  {acc:20s} -> {format_accession_with_type(acc)}")

Accession Type Classification Test:
--------------------------------------------------
  SAMN08031253         -> SAMN08031253[BioSample]
  PRJNA448609          -> PRJNA448609[BioProject]
  SRR12967252          -> SRR12967252[SRA]
  NC_001371            -> NC_001371[RefSeq]
  GCA_000001405.1      -> GCA_000001405.1[Assembly]
  JAHCQI000000000      -> JAHCQI000000000[WGS]
  AAR18814.1           -> AAR18814.1[Protein]
  AB123456             -> AB123456[GenBank]


## Helper Functions

In [15]:
def load_discrepancies_csv(filepath: str) -> Dict[str, List[str]]:
    """Load discrepancies CSV and extract PMCIDs with new accessions."""
    discrepancies = {}

    with open(filepath, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter='|')

        for row in reader:
            pmcid = row['pmcid']
            new_accessions_str = row.get('new_accessions', '').strip()

            if not new_accessions_str:
                continue

            new_accessions = [a.strip() for a in new_accessions_str.split(',') if a.strip()]

            if new_accessions:
                discrepancies[pmcid] = new_accessions

    return discrepancies


def load_gt_json(pmcid: str, gt_folder: str) -> Optional[Dict]:
    """Load a ground truth JSON file."""
    json_path = Path(gt_folder) / f"{pmcid}.json"

    if not json_path.exists():
        return None

    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        print(f"  Error loading {pmcid}.json: {e}")
        return None


def save_gt_json(pmcid: str, data: Dict, gt_folder: str) -> bool:
    """Save updated ground truth JSON file."""
    json_path = Path(gt_folder) / f"{pmcid}.json"

    try:
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4, ensure_ascii=False)
        return True
    except Exception as e:
        print(f"  Error saving {pmcid}.json: {e}")
        return False

## Main Update Function

In [16]:
def update_ground_truth_files(
    discrepancies_csv: str = DISCREPANCIES_CSV,
    gt_folder: str = GROUND_TRUTH_FOLDER,
    output_log: str = OUTPUT_LOG,
    excluded_pmcids: Set[str] = EXCLUDED_PMCIDS,
    dry_run: bool = False
) -> List[Dict]:
    """
    Update ground truth JSON files with new accessions.

    Parameters:
        dry_run: If True, preview only (no file modifications)
    """
    print("=" * 80)
    print("UPDATE GROUND TRUTH JSON FILES")
    print("=" * 80)

    if dry_run:
        print("\n*** DRY RUN MODE - No files will be modified ***\n")

    # Load discrepancies
    print(f"Loading discrepancies from: {discrepancies_csv}")
    discrepancies = load_discrepancies_csv(discrepancies_csv)
    print(f"PMCIDs with new accessions: {len(discrepancies)}")

    # Filter excluded
    pmcids_to_update = {k: v for k, v in discrepancies.items() if k not in excluded_pmcids}
    print(f"PMCIDs to update (after exclusions): {len(pmcids_to_update)}")

    # Process each PMCID
    results = []
    updated_count = 0
    skipped_count = 0
    error_count = 0

    print("\n" + "-" * 80)

    for pmcid, new_accessions in pmcids_to_update.items():
        print(f"\n[{pmcid}] {len(new_accessions)} new accession(s)")

        # Load existing JSON
        gt_data = load_gt_json(pmcid, gt_folder)

        if gt_data is None:
            print(f"  SKIPPED: JSON file not found")
            skipped_count += 1
            results.append({
                "pmcid": pmcid,
                "status": "SKIPPED",
                "reason": "JSON not found",
                "added_count": 0
            })
            continue

        # Get existing accessions
        existing_accessions = gt_data.get('merge_accession_number', [])

        # Extract accession part only for comparison
        existing_acc_set = set()
        for acc_with_type in existing_accessions:
            acc_only = re.sub(r'\[[^\]]+\]$', '', acc_with_type)
            existing_acc_set.add(acc_only.upper())

        # Format and filter duplicates
        accessions_to_add = []
        for acc in new_accessions:
            if acc.upper() not in existing_acc_set:
                formatted = format_accession_with_type(acc)
                accessions_to_add.append(formatted)
                print(f"  + {formatted}")
            else:
                print(f"  ~ {acc} (already exists)")

        if not accessions_to_add:
            print(f"  SKIPPED: All already exist")
            skipped_count += 1
            results.append({
                "pmcid": pmcid,
                "status": "SKIPPED",
                "reason": "All exist",
                "added_count": 0
            })
            continue

        # Add new accessions
        gt_data['merge_accession_number'] = existing_accessions + accessions_to_add

        # Save
        if dry_run:
            print(f"  [DRY RUN] Would add {len(accessions_to_add)}")
            updated_count += 1
        else:
            if save_gt_json(pmcid, gt_data, gt_folder):
                print(f"  UPDATED: Added {len(accessions_to_add)}")
                updated_count += 1
            else:
                error_count += 1

        results.append({
            "pmcid": pmcid,
            "status": "UPDATED" if not dry_run else "DRY_RUN",
            "reason": "",
            "new_accessions": ",".join(accessions_to_add),
            "added_count": len(accessions_to_add)
        })

    # Write log
    if not dry_run:
        with open(output_log, 'w', newline='', encoding='utf-8') as f:
            fieldnames = ['pmcid', 'status', 'reason', 'added_count', 'new_accessions']
            writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter='|')
            writer.writeheader()
            writer.writerows(results)
        print(f"\nLog saved to: {output_log}")

    # Summary
    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print(f"Processed: {len(pmcids_to_update)}")
    print(f"Updated: {updated_count}")
    print(f"Skipped: {skipped_count}")
    print(f"Errors: {error_count}")

    return results

## Step 1: Preview (Dry Run)

In [17]:
# Preview what changes would be made
results = update_ground_truth_files(dry_run=True)

UPDATE GROUND TRUTH JSON FILES

*** DRY RUN MODE - No files will be modified ***

Loading discrepancies from: /content/drive/MyDrive/AI6129/gt_discrepancies_detailed.csv
PMCIDs with new accessions: 150
PMCIDs to update (after exclusions): 150

--------------------------------------------------------------------------------

[PMC7643746] 7 new accession(s)
  + AL513382[GenBank]
  + B12730[GenBank]
  + B22992[GenBank]
  + B24112[GenBank]
  + B32375[GenBank]
  + B39348[GenBank]
  + MED00290[Protein]
  [DRY RUN] Would add 7

[PMC6817352] 63 new accession(s)
  + CP006693[GenBank]
  + CP012144[GenBank]
  + CP018655[GenBank]
  + KR350635[GenBank]
  + KU641443[GenBank]
  + KY656601[GenBank]
  + MG663456[GenBank]
  + MG663457[GenBank]
  + MG663458[GenBank]
  + MG663459[GenBank]
  + MG663460[GenBank]
  + MG663461[GenBank]
  + MG663462[GenBank]
  + MG663463[GenBank]
  + MG663464[GenBank]
  + MG663465[GenBank]
  + MG663466[GenBank]
  + MG663467[GenBank]
  + MG663468[GenBank]
  + MG663469[GenBank]


## Step 2: Apply Updates

**Run this cell ONLY after reviewing the preview above.**

This will modify the ground truth JSON files!

In [19]:
# UNCOMMENT TO APPLY UPDATES
results = update_ground_truth_files(dry_run=False)

UPDATE GROUND TRUTH JSON FILES
Loading discrepancies from: /content/drive/MyDrive/AI6129/gt_discrepancies_detailed.csv
PMCIDs with new accessions: 150
PMCIDs to update (after exclusions): 150

--------------------------------------------------------------------------------

[PMC7643746] 7 new accession(s)
  + AL513382[GenBank]
  + B12730[GenBank]
  + B22992[GenBank]
  + B24112[GenBank]
  + B32375[GenBank]
  + B39348[GenBank]
  + MED00290[Protein]
  UPDATED: Added 7

[PMC6817352] 63 new accession(s)
  + CP006693[GenBank]
  + CP012144[GenBank]
  + CP018655[GenBank]
  + KR350635[GenBank]
  + KU641443[GenBank]
  + KY656601[GenBank]
  + MG663456[GenBank]
  + MG663457[GenBank]
  + MG663458[GenBank]
  + MG663459[GenBank]
  + MG663460[GenBank]
  + MG663461[GenBank]
  + MG663462[GenBank]
  + MG663463[GenBank]
  + MG663464[GenBank]
  + MG663465[GenBank]
  + MG663466[GenBank]
  + MG663467[GenBank]
  + MG663468[GenBank]
  + MG663469[GenBank]
  + MG663470[GenBank]
  + MG663471[GenBank]
  + MG663472

## View Update Log

In [20]:
# View log after applying updates
import pandas as pd

try:
    df_log = pd.read_csv(OUTPUT_LOG, delimiter='|')
    print(f"Total entries: {len(df_log)}")
    print(f"\nUpdated PMCIDs:")
    display(df_log[df_log['status'] == 'UPDATED'])
except FileNotFoundError:
    print("Log file not found. Run Step 2 first.")

Total entries: 150

Updated PMCIDs:


,pmcid,status,reason,added_count,new_accessions
0,PMC7643746,UPDATED,NaN,7,"AL513382[GenBank],B12730[GenBank],B22992[GenBa..."
1,PMC6817352,UPDATED,NaN,63,"CP006693[GenBank],CP012144[GenBank],CP018655[G..."
2,PMC8300843,UPDATED,NaN,1,PMC89009[Protein]
3,PMC6118388,UPDATED,NaN,15,"AAR18814.1[Protein],CP013673[GenBank],CP016573..."
4,PMC5075293,UPDATED,NaN,1,P11101[GenBank]
...,...,...,...,...,...
145,PMC6430326,UPDATED,NaN,1,MRA01687[Protein]
146,PMC8175864,UPDATED,NaN,19,"CP014051[GenBank],CP039562[GenBank],CP055697[G..."
147,PMC8082823,UPDATED,NaN,4,"D23580[GenBank],GCF_000006945[Assembly],GCF_00..."
148,PMC8913728,UPDATED,NaN,10,"N16961[GenBank],NC_003315[RefSeq],NC_009237[Re..."
